# Architectures as blocks

Tensor shapes, residuals, normalization, and gradient checks.

> All examples run on CPU without model downloads and use synthetic data. They explain mechanisms; real training quality must be evaluated separately.

## What to understand

A block transforms a tensor. A residual adds the input to the block output, so shapes must match. LayerNorm normalizes features; an MLP transforms representations. Attention lets positions exchange information and requires the correct causal mask.

Model Builder supports graph editing, block references, shape analysis, and training handoffs for supported LLM/VLM/VLA architectures. Arbitrary graphs are not universally compatible: modality, causality, and output shapes matter. This exercise probes a residual MLP, not the complete Builder validator.

In [ ]:
%matplotlib inline
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(42)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False, 'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| device: CPU')

## Run the experiment

Run cells in order. Edit the code; Shift + Enter runs the current cell.

In [ ]:
class ResidualMLP(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.norm = nn.LayerNorm(width)
        self.mlp = nn.Sequential(nn.Linear(width, width * 2), nn.GELU(), nn.Linear(width * 2, width))
    def forward(self, x):
        return x + self.mlp(self.norm(x))
block = ResidualMLP(16)
x = torch.randn(2, 8, 16, requires_grad=True)
y = block(x); y.square().mean().backward()
print('input:', tuple(x.shape), 'output:', tuple(y.shape))
print('parameters:', sum(p.numel() for p in block.parameters()))
print('input gradient norm:', x.grad.norm().item())
assert x.shape == y.shape and torch.isfinite(x.grad).all()

## Try it yourself

Change the final Linear to output width 8 and inspect the residual error. Restore the shape and compare gradient norms with and without LayerNorm.

Save a notebook copy and record what you changed, expected, and measured.

## Continue in Studio

The workspace link above the lesson opens related tools. This notebook does not launch large training jobs. Before real training, check data, resources, and method support for the chosen model.

## Sources

- [Residual Learning](https://arxiv.org/abs/1512.03385)
- [Layer Normalization](https://arxiv.org/abs/1607.06450)